In [1]:
from scripts.dataset import X_train, X_test, y_train, y_test, groups_train, groups_test, numeric_features, categorical_features

DATASET
Total samples:       6,881
Total trajectories:  750
TRAIN / TEST SPLIT
Train samples:       5,491
Test samples:        1,390
Train trajectories:  600
Test trajectories:   150
TRAIN LABEL DISTRIBUTION
       count  percentage
label                   
-1      1457       26.53
 0       231        4.21
 1      3803       69.26
TEST LABEL DISTRIBUTION
       count  percentage
label                   
-1       402       28.92
 0        56        4.03
 1       932       67.05
LEAKAGE CHECK
Overlapping trajectories: 0
X / y / groups alignment: OK
Trajectory split:          OK


In [2]:
def build_transformer_frame(X, y):
    df = X.copy()

    df["text"] = (
        "[CONTEXT]\n"
        + df["context_text"].fillna("")
        + "\n\n[CURRENT]\n"
        + df["current_text"].fillna("")
    )

    # Hugging Face classification labels should be 0..num_labels-1
    label_map = {
        -1: 0,
         0: 1,
         1: 2,
    }

    df["labels"] = (
        y.reset_index(drop=True)
        .map(label_map)
        .astype(int)
    )

    return df[["text", "labels"]].reset_index(drop=True)

In [3]:
train_df = build_transformer_frame(
    X_train.reset_index(drop=True),
    y_train.reset_index(drop=True),
)

test_df = build_transformer_frame(
    X_test.reset_index(drop=True),
    y_test.reset_index(drop=True),
)

print(train_df.shape)
print(test_df.shape)

print(train_df["labels"].value_counts())

(5491, 2)
(1390, 2)
labels
2    3803
0    1457
1     231
Name: count, dtype: int64


In [4]:
from datasets import Dataset

hf_train = Dataset.from_pandas(
    train_df,
    preserve_index=False,
)

hf_test = Dataset.from_pandas(
    test_df,
    preserve_index=False,
)

In [5]:
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

In [6]:
MAX_LENGTH = 512

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )


tokenized_train = hf_train.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"],
)

tokenized_test = hf_test.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"],
)

Map:   0%|          | 0/5491 [00:00<?, ? examples/s]

Map:   0%|          | 0/1390 [00:00<?, ? examples/s]

In [7]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

In [8]:
from transformers import AutoModelForSequenceClassification

id2label = {
    0: "-1",
    1: "0",
    2: "+1",
}

label2id = {
    "-1": 0,
    "0": 1,
    "+1": 2,
}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [9]:
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)


def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1,
    )

    return {
        "accuracy": accuracy_score(
            labels,
            predictions,
        ),

        "macro_f1": f1_score(
            labels,
            predictions,
            average="macro",
        ),

        "error_precision": precision_score(
            labels,
            predictions,
            labels=[0],  # original -1
            average="macro",
            zero_division=0,
        ),

        "error_recall": recall_score(
            labels,
            predictions,
            labels=[0],
            average="macro",
            zero_division=0,
        ),

        "error_f1": f1_score(
            labels,
            predictions,
            labels=[0],
            average="macro",
            zero_division=0,
        ),
    }

In [10]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./distilbert-agent-reliability",

    learning_rate=2e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,

    num_train_epochs=3,

    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    logging_steps=50,

    seed=42,

    report_to="none",
)

In [12]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,

    processing_class=tokenizer,
    data_collator=data_collator,

    compute_metrics=compute_metrics,
)

In [13]:
trainer.train()

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
transformer_results = trainer.evaluate()

transformer_results

In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
)

pred_output = trainer.predict(
    tokenized_test
)

predictions = np.argmax(
    pred_output.predictions,
    axis=-1,
)

labels = pred_output.label_ids

print(
    classification_report(
        labels,
        predictions,
        labels=[0, 1, 2],
        target_names=["-1", "0", "+1"],
        zero_division=0,
        digits=4,
    )
)

print(
    confusion_matrix(
        labels,
        predictions,
        labels=[0, 1, 2],
    )
)